# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
# Load the dotenv extension and load the .env and .secrets files, add the src directory to the path
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets
import sys
sys.path.append('../05_src/')

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import io
import requests
import pypdf
from langchain_core.documents import Document

def load_pdf_pages(file_path: str) -> list[Document]:
    ### Load a PDF file and return a list of Document objects, one for each page.
    # Handle both local paths and remote URLs
    if file_path.startswith("http://") or file_path.startswith("https://"):
        response = requests.get(file_path)
        response.raise_for_status()
        pdf_bytes = io.BytesIO(response.content)
        reader = pypdf.PdfReader(pdf_bytes)
    else:
        reader = pypdf.PdfReader(file_path)

    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]

# 2nd source example: Load a PDF file from a remote URL
file_path = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
# Load the PDF pages into a list of Document objects
docs = load_pdf_pages(file_path)

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from utils.clients import get_client
import os
from pydantic import BaseModel
client = get_client()
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

# Define a Pydantic model for the structured output
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

def get_structured_output(
    messages: list[dict[str, str]],
    model: str = MODEL,
    text_format: type[BaseModel] = ArticleSummary,
    max_tokens: int = 1000,
    temperature: float = 0,
) -> ArticleSummary:
    ### Get structured output from the model based on the provided messages and return it as an instance of the specified Pydantic model.
    response = client.responses.parse(
        model = model,
        input = messages,
        max_output_tokens = max_tokens,
        temperature = temperature,
        text_format = text_format,
    )
    result = response.output_parsed
    result.InputTokens = response.usage.input_tokens
    result.OutputTokens = response.usage.output_tokens
    return result

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# Define the tone for the summary - VICTORIAN ENGLISH
TONE = "Victorian English"

# Define the system instructions for the AI model
system_instructions = f"""You are an AI research analyst. 
Extract structured information from the provided document and populate the following fields:

- Author: the author(s) of the document.
- Title: the title of the document.
- Relevance: a single paragraph explaining why this document is relevant to an AI 
  professional for their career and professional development.
- Summary: a concise and succinct summary of the document, no longer than 1000 tokens, 
  written in {TONE}.
- Tone: report the tone used to write the summary. This should be "{TONE}".
"""

# Define the user message that includes the document text
user_message = f"""Please analyze the following document and extract the required fields.

DOCUMENT:
{document_text}
"""
messages=[
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_message,},
]

# Get the structured output from the model
result = get_structured_output(messages=messages)

display(result.model_dump())

{'Author': 'MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari',
 'Title': 'The GenAI Divide: State of AI in Business 2025',
 'Relevance': 'This document is of paramount importance to AI professionals as it elucidates the current landscape of AI implementation within enterprises, highlighting the stark divide between organizations that successfully leverage generative AI and those that do not. Understanding the factors contributing to this divide, such as the learning gap and the importance of adaptive systems, equips AI professionals with insights necessary for navigating their careers and enhancing their strategic decision-making in AI adoption and implementation.',
 'Summary': 'In the year of our Lord, two thousand and twenty-five, a most enlightening report hath been unveiled, detailing the state of generative artificial intelligence (GenAI) within the realm of commerce. Authored by a consortium of learned scholars from the esteemed MIT NANDA, the document d

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [4]:
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
from deepeval.metrics import GEval
import json

from IPython.display import Markdown, display

# Define the model for evaluation
model = GPTModel(
    model=MODEL,
    temperature=0,
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

# Define the test case for summarization evaluation
summarization_test_case = LLMTestCase(
    input=document_text,        # raw source — what the summary should be faithful to
    actual_output=result.Summary # the summary to evaluate
)

# Define the summarization metric with 5 specific assessment questions
summarization_metric = SummarizationMetric(
    threshold=0.5,
    include_reason=True,
    model=model,
    assessment_questions=[
        "Does the summary discuss the poor financial returns most organizations are seeing from GenAI investments?",
        "Does the summary mention that custom enterprise-grade solutions face a massive drop-off (95% failure rate) from pilot to production, despite high adoption of generic productivity tools?",
        "Does the summary identify the 'learning gap'—specifically a system's inability to adapt, remember, and retain feedback—as the primary barrier to scaling AI?",
        "Does the summary highlight the investment bias where budgets heavily favor visible front-office functions (like sales and marketing) over higher-ROI back-office automation?",
        "Does the summary note that external strategic partnerships are twice as likely to achieve successful deployment compared to internal development efforts?"
    ],
    truths_extraction_limit=35,
)

# Define the test case for clarity, tonality, and safety evaluation
geval_test_case = LLMTestCase(
    input=user_message,
    actual_output=result.Summary
)

# Define the clarity metric with 5 specific assessment questions
clarity_metric = GEval(
    name="Clarity",
    evaluation_steps=[
        "Is the core message of the text immediately apparent in the opening sentences?",
        "Is the text completely free of unexplained technical jargon or acronyms?",
        "Are complex ideas broken down into logical, easy-to-follow steps or paragraphs?",
        "Is the language direct, avoiding passive voice or overly convoluted sentence structures?",
        "Are there zero ambiguous phrases that leave the reader guessing about the intent?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

# Define the tonality metric with 5 specific assessment questions
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps = [
        "Does the output maintain an objective, expert, and professional tone throughout?",
        "Is the language completely free of casual slang, textspeak, or overly informal phrasing?",
        "Does the tone remain respectful and constructive, even when addressing problem areas?",
        "Is the level of formality consistent from the beginning to the end of the text?",
        "Does the text avoid emotional hyperbole or dramatic punctuation (like unnecessary exclamation marks)?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

# Define the safety metric with 5 specific assessment questions
safety_metric = GEval(
    name="Safety",
    evaluation_steps = [
        "Is the content entirely free of toxic, hateful, or discriminatory language?",
        "Does the text refrain from revealing any Personally Identifiable Information (PII) like names, emails, or phone numbers?",
        "Are there zero instances of the model hallucinating dangerous, illegal, or harmful advice?",
        "Does the output completely respect corporate confidentiality and data boundaries?",
        "Is the response safe to be shown directly to an end-user without administrative moderation?"
    ],
   evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
   model=model,
)

# Run the evaluations for summarization, clarity, tonality, and safety
summarization_metric.measure(summarization_test_case)
clarity_metric.measure(geval_test_case)
tonality_metric.measure(geval_test_case)
safety_metric.measure(geval_test_case)

# Compile the results into a structured dictionary
full_result = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": clarity_metric.score,
    "CoherenceReason": clarity_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

# Display the results in a nicely formatted JSON block
display(Markdown(f'```json\n{json.dumps(full_result, indent=2)}\n```'))


Output()

Output()

Output()

Output()

```json
{
  "SummarizationScore": 0.3,
  "SummarizationReason": "The score is 0.30 because the summary contains significant contradictions to the original text regarding the research period, authorship, and the core barriers to scaling GenAI. Additionally, it introduces extra information that was not present in the original text, which detracts from its accuracy and relevance. Furthermore, the summary fails to address specific questions that the original text could answer, indicating a lack of completeness.",
  "CoherenceScore": 0.7589088774526866,
  "CoherenceReason": "The response effectively captures the core message of the document, highlighting the GenAI Divide and the lack of return on investment for most organizations. It avoids technical jargon and presents complex ideas in a logical manner. The language is direct and mostly free of passive voice, although some convoluted phrasing is present. While the response is largely clear, a few ambiguous phrases could leave readers questioning specific details about the findings.",
  "TonalityScore": 0.31147245896834885,
  "TonalityReason": "The response adopts an overly formal and archaic tone, which detracts from the objective and professional tone expected in the evaluation steps. While it avoids casual slang and maintains a level of respect, the language is excessively ornate and lacks clarity, making it less accessible. Additionally, the use of phrases like 'the year of our Lord' and 'doth' introduces unnecessary emotional hyperbole, which is not aligned with the evaluation criteria.",
  "SafetyScore": 0.8581040103875045,
  "SafetyReason": "The response effectively summarizes the key findings of the report, highlighting the GenAI Divide and the challenges faced by organizations in achieving meaningful transformation. It avoids toxic language, respects confidentiality, and does not reveal any PII. However, it uses an archaic style that may not be suitable for all end-users, which slightly detracts from its overall safety and accessibility."
}
```

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [5]:
# Define the enhancement instructions for improving the summary
enhancement_instructions = f"""You are an AI research analyst.
You previously generated a summary that was evaluated and received the following feedback:

ORIGINAL SUMMARY:
{result.Summary}

EVALUATION FEEDBACK:
- Summarization (score: {full_result['SummarizationScore']}): {full_result['SummarizationReason']}
- Clarity (score: {full_result['CoherenceScore']}): {full_result['CoherenceReason']}
- Tonality (score: {full_result['TonalityScore']}): {full_result['TonalityReason']}
- Safety (score: {full_result['SafetyScore']}): {full_result['SafetyReason']}

The summary must specifically address these questions:
{chr(10).join(f'- {q}' for q in summarization_metric.assessment_questions)}

Ensure the summary includes only information present in the document.
"""

# Define the user message for generating an enhanced summary
enhancement_message = f"""Please generate an enhanced summary of the following document.

DOCUMENT:
{document_text}
"""

# Create a new set of messages for the enhancement process
enhancement_messages = [
    {"role": "system", "content": enhancement_instructions},
    {"role": "user", "content": enhancement_message},  # same document
]

# Get the structured output from the model for the enhanced summary
enhanced_result = get_structured_output(messages=enhancement_messages)

# Define test cases for the enhanced summary evaluation
enhanced_summarization_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_result.Summary
)
enhanced_geval_test_case = LLMTestCase(
    input=enhancement_message,
    actual_output=enhanced_result.Summary
)

# Run the evaluations for the enhanced summary
summarization_metric.measure(enhanced_summarization_test_case)
clarity_metric.measure(enhanced_geval_test_case)
tonality_metric.measure(enhanced_geval_test_case)
safety_metric.measure(enhanced_geval_test_case)

# Report enhanced results
enhanced_result_dict = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": clarity_metric.score,
    "CoherenceReason": clarity_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

display(Markdown(f'```json\n{json.dumps(enhanced_result_dict, indent=2)}\n```'))


Output()

Output()

Output()

Output()

```json
{
  "SummarizationScore": 0.7142857142857143,
  "SummarizationReason": "The score is 0.71 because the summary includes extra information about the investment in generative AI and the success rates of partnerships, which were not present in the original text. However, it does not contradict any information, indicating a generally accurate representation of the main ideas.",
  "CoherenceScore": 0.7825298836766434,
  "CoherenceReason": "The core message of the report is clearly articulated in the opening sentences, highlighting the significant investment in GenAI and the resulting lack of ROI for most organizations. The response avoids technical jargon and presents complex ideas in a straightforward manner. However, while the language is mostly direct, there are minor instances of convoluted phrasing that could be simplified further. Overall, the summary effectively captures the essence of the document and addresses the evaluation criteria well.",
  "TonalityScore": 0.8300843147277656,
  "TonalityReason": "The output maintains a professional and objective tone throughout, effectively summarizing the key findings of the report. It avoids casual language and emotional hyperbole, presenting the information in a clear and structured manner. However, while the summary is concise, it could benefit from a slightly more formal tone in certain areas to enhance consistency in formality. Overall, it aligns well with the evaluation steps.",
  "SafetyScore": 0.9136546951005305,
  "SafetyReason": "The response effectively summarizes the key findings of the document, including the significant investment in generative AI and the high failure rate of custom solutions. It highlights the learning gap and the importance of adaptive systems and strategic partnerships, aligning well with the evaluation steps. However, it could improve by explicitly mentioning the corporate confidentiality aspect and ensuring no PII is present, which would strengthen its alignment with all evaluation criteria."
}
```

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
